In [1]:
import pandas as pd

excel_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(excel_path, engine='openpyxl')

df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,101.71.130.99,2026-02-25,Address,133,3,0.992712,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",NaN,NaN,530,521,469,medium,[2026-02-25] Severity: medium. VT score: 10. T...
1,103.120.116.162,2026-02-25,Address,29,3,0.998411,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,770,885,538,high,[2026-02-25] Severity: high. VT score: 4. Top ...
2,103.172.81.132,2026-02-25,Address,35,0,0.998082,0,0,"CMS, FDA, HHS, NIH, VA",Incident:5629499550000872;Incident:56294995470...,"Emissary Panda, Judgment Panda, UNC6337/STORM-...",180,288,180,low,[2026-02-25] Severity: low. VT score: 2. Top d...
3,103.216.220.19,2026-02-23,Address,19,5,0.998959,0,0,"CMS, DHA, OS, VA",NaN,UNC5537,170,464,440,medium,[2026-02-25] Severity: medium. VT score: 4. To...
4,103.243.26.49,2026-02-25,Address,36,3,0.998027,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,469,524,high,[2026-02-25] Severity: high. VT score: 5. Top ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2215,5.79.110.170,2025-12-23,Address,6,0,0.999671,1,0,"CDC, CMS, HHS",Incident:6755399448003491;Incident:529312,NaN,170,363,45,low,Severity: low. VT score: 0. Top drivers: Incid...
2216,146.70.246.116,2026-01-19,Address,3,0,0.999836,1,0,VA,Incident:6755399478001848,NaN,180,368,41,low,Severity: low. VT score: 0. Top drivers: Incid...
2217,198.199.103.243,2026-01-16,Address,1,0,0.999945,1,0,VA,Incident:6755399478001848,NaN,180,368,41,low,Severity: low. VT score: 0. Top drivers: Incid...
2218,rosesci.com,2026-01-08,Host,17,0,0.999068,0,0,NIH,NaN,NaN,170,351,30,low,Severity: low. Could not pull VT score to calc...


In [2]:
import glob
import os

csv_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports"
csv_pattern = "full_daily_report_*.csv"
csv_paths = sorted(glob.glob(f"{csv_dir}\\{csv_pattern}"))

print(f"Found {len(csv_paths)} CSV files")

dfs = []
loaded_files = []
skipped_files = []

for path in csv_paths:
    try:
        # Check if file exists and has content
        if not os.path.exists(path):
            skipped_files.append((path, "File not found"))
            continue
            
        file_size = os.path.getsize(path)
        if file_size == 0:
            skipped_files.append((path, "Empty file"))
            continue
        
        # Read CSV
        df_temp = pd.read_csv(path)
        
        # Check if DataFrame is empty
        if df_temp.empty:
            skipped_files.append((path, "No data rows"))
            continue
        
        dfs.append(df_temp)
        loaded_files.append((path, len(df_temp)))
        
    except Exception as e:
        skipped_files.append((path, f"Error: {str(e)}"))
        continue

print(f"\nLoaded {len(loaded_files)} files successfully")
print(f"Skipped {len(skipped_files)} files")

if skipped_files:
    print("\nSkipped files:")
    for path, reason in skipped_files[:10]:  # Show first 10
        print(f"  {os.path.basename(path)}: {reason}")
    if len(skipped_files) > 10:
        print(f"  ... and {len(skipped_files) - 10} more")

if dfs:
    # Use sort=False to preserve column order, handle mismatches
    df_daily_reports = pd.concat(dfs, ignore_index=True, sort=False)
    print(f"\nTotal rows loaded: {len(df_daily_reports)}")
    print(f"Total columns: {len(df_daily_reports.columns)}")
    
    # Fill missing values with "unknown" for object/string columns
    for c in df_daily_reports.select_dtypes(include=['object']).columns:
        df_daily_reports[c] = df_daily_reports[c].fillna('unknown')
    
    # Fill missing numeric values with median per indicator
    import numpy as np
    numeric_cols = df_daily_reports.select_dtypes(include=[np.number]).columns.tolist()
    if 'Indicator' in df_daily_reports.columns and numeric_cols:
        for col in numeric_cols:
            # Calculate median per indicator group
            median_by_indicator = df_daily_reports.groupby('Indicator')[col].transform('median')
            # Fill missing values with the median for that indicator
            df_daily_reports[col] = df_daily_reports[col].fillna(median_by_indicator)
            # If indicator group has all NaN, fill with overall median
            df_daily_reports[col] = df_daily_reports[col].fillna(df_daily_reports[col].median())
    
else:
    print("\nWARNING: No data was loaded from any files!")
    df_daily_reports = pd.DataFrame()

df_daily_reports

Found 233 CSV files

Loaded 233 files successfully
Skipped 0 files

Total rows loaded: 2238086
Total columns: 17


,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner,FileDate,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,20250616,unknown,unknown,0.0,unknown,unknown
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,20250616,unknown,unknown,0.0,unknown,unknown
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,20250616,unknown,unknown,0.0,unknown,unknown
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,20250616,unknown,unknown,0.0,unknown,unknown
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,20250616,unknown,unknown,0.0,unknown,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2238081,tester.com,0,0.0,0.0,0.15%,7-Day: Low confidence,0.27%,14-Day: Low confidence,12.73%,30-Day: Low confidence,VA,20260225,34.08%,45-Day: Possibly active,0.0,0.02%,1-Day: Low confidence
2238082,www.citiquartz.com,0,1.0,1.0,54.41%,7-Day: Possibly active,61.37%,14-Day: Possibly active,77.08%,30-Day: Possibly active,VA,20260225,66.74%,45-Day: Highly likely,0.0,25.71%,1-Day: Low confidence
2238083,www.deepseek.com,0,5.0,21.0,99.51%,7-Day: Highly likely,99.95%,14-Day: Highly likely,100.0%,30-Day: Highly likely,VA,20260225,83.38%,45-Day: Highly likely,0.0,36.88%,1-Day: Low confidence
2238084,www.deepseek.com.cdn.cloudflare.net,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.03%,30-Day: Low confidence,VA,20260225,0.25%,45-Day: Low confidence,0.0,0.01%,1-Day: Low confidence


In [3]:
import numpy as np

# Aggregate by Indicator and take averages for everything else
def parse_pct(s):
    """Parse '85%' or '85.2%' to float."""
    if pd.isna(s) or s == '':
        return np.nan
    try:
        return float(str(s).strip().rstrip('%'))
    except ValueError:
        return np.nan

# Drop file-related columns (e.g. FileDate) before aggregating
df_daily_reports = df_daily_reports.drop(columns=['FileDate'], errors='ignore')

numeric_cols = df_daily_reports.select_dtypes(include=[np.number]).columns.tolist()
# Separate Probability (percentages) from Confidence (descriptive strings)
prob_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and ('Probability' in c or c == 'ensemble_45d')]
conf_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and 'Confidence' in c]
obj_cols = [c for c in df_daily_reports.columns if c != 'Indicator' and c not in numeric_cols and c not in prob_cols and c not in conf_cols]

agg_dict = {c: 'mean' for c in numeric_cols}
agg_dict.update({c: 'first' for c in obj_cols})
# Confidence columns use mode (most frequent label)
def make_mode_agg():
    def mode_agg(x):
        mode_vals = x.mode()
        if len(mode_vals) > 0:
            return mode_vals[0]
        elif len(x) > 0:
            return x.iloc[0]
        else:
            return 'unknown'
    return mode_agg

for c in conf_cols:
    agg_dict[c] = make_mode_agg()
# Only parse Probability columns as percentages
def make_pct_agg():
    def pct_agg(x):
        vals = x.apply(parse_pct)
        if vals.notna().any():
            return f"{vals.mean():.2f}%"
        else:
            return "unknown"
    return pct_agg

for c in prob_cols:
    agg_dict[c] = make_pct_agg()

df_daily_agg = df_daily_reports.groupby('Indicator').agg(agg_dict).reset_index()

# Round numeric columns to 2 decimal places
for c in df_daily_agg.select_dtypes(include=[np.number]).columns:
    df_daily_agg[c] = df_daily_agg[c].round(2)

# Rename columns to reflect aggregation method
rename_dict = {}
# Numeric columns use mean (average)
for c in numeric_cols:
    rename_dict[c] = f"{c} (Average)"
# Probability columns use mean (average) of percentages
for c in prob_cols:
    if c == 'ensemble_45d':
        rename_dict[c] = "Probability: 45-Day (Average)"
    else:
        rename_dict[c] = f"{c} (Average)"
# Object columns use first
for c in obj_cols:
    rename_dict[c] = f"{c} (First)"
# Confidence columns use mode
for c in conf_cols:
    rename_dict[c] = f"{c} (Mode)"

df_daily_agg = df_daily_agg.rename(columns=rename_dict)

del df_daily_reports  # drop raw file data
df_daily_agg

,Indicator,Observed Today (Average),Frequency (7d) (Average),Frequency (30d) (Average),Frequency (1d) (Average),Partner (First),Confidence: 7-Day (Mode),Confidence: 14-Day (Mode),Confidence: 30-Day (Mode),Confidence: 45-Day (Mode),Confidence: 1-Day (Mode),Probability: 7-Day (Average),Probability: 14-Day (Average),Probability: 30-Day (Average),Probability: 45-Day (Average),Probability: 1-Day (Average)
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0.00,0.00,0.84,0.00,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Highly likely,1-Day: Low confidence,16.58%,33.45%,48.79%,52.79%,0.57%
1,1-you.njalla.no,0.01,0.45,1.94,0.01,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,unknown,25.66%,43.13%,64.95%,50.55%,10.23%
2,1.192.18.4,0.01,0.21,0.95,0.00,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,unknown,15.94%,29.39%,46.53%,43.33%,0.01%
3,1.204.166.3,0.00,0.12,0.71,0.00,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,12.10%,24.36%,48.52%,41.00%,0.08%
4,1.234.83.26,0.38,3.65,14.48,0.38,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,83.04%,90.74%,95.63%,78.75%,42.34%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5685,ymath.yonsei.ac.kr,0.00,0.00,0.03,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,1-Day: Low confidence,0.17%,1.02%,2.64%,12.94%,0.02%
5686,yotpo-static.com,0.00,0.02,0.08,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Low confidence,1-Day: Low confidence,1.39%,1.69%,5.26%,13.81%,0.04%
5687,yourpensionmeeting.com,0.33,2.83,11.30,0.19,VA,7-Day: Low confidence,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,unknown,54.94%,65.24%,77.88%,64.91%,29.93%
5688,zerocap.com,0.00,0.00,0.00,0.00,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,0.00%,0.00%,0.00%,unknown,unknown


In [4]:
# Check the Indicator column in df (threat scores)
print("Threat Assessment Scores (df) columns:")
print(df.columns.tolist())
print(f"\ndf shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())


Threat Assessment Scores (df) columns:
['Indicator', 'Last Observed', 'Indicator Type', 'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'Partners', 'incidents/events', 'Threat Actor', 'CAL Score', 'ThreatConnect Score', 'HTOC Threat Score', 'Severity', 'Explanation']

df shape: (2220, 16)

First few rows:
         Indicator Last Observed Indicator Type  Observation Yearly Count  \
0    101.71.130.99    2026-02-25        Address                       133   
1  103.120.116.162    2026-02-25        Address                        29   
2   103.172.81.132    2026-02-25        Address                        35   
3   103.216.220.19    2026-02-23        Address                        19   
4    103.243.26.49    2026-02-25        Address                        36   

   ThreatConnect Rating  Observation Penalty Multiplier  Botnet Flag  \
0                     3                        0.992712            0   
1             

In [5]:
# Merge df (threat scores) with df_daily_agg on Indicator
# Use inner join to keep only records that have matching scores
print("Before merge:")
print(f"  df_daily_agg: {len(df_daily_agg)} rows")
print(f"  df (threat scores): {len(df)} rows")

# Merge on 'Indicator' column - this will keep only matching records
df_merged = df_daily_agg.merge(df, on='Indicator', how='inner')

print(f"\nAfter merge (inner join):")
print(f"  df_merged: {len(df_merged)} rows")
print(f"\nColumns in merged dataframe: {df_merged.columns.tolist()}")
print(f"\nShape: {df_merged.shape}")

df_merged


Before merge:
  df_daily_agg: 5690 rows
  df (threat scores): 2220 rows

After merge (inner join):
  df_merged: 2035 rows

Columns in merged dataframe: ['Indicator', 'Observed Today (Average)', 'Frequency (7d) (Average)', 'Frequency (30d) (Average)', 'Frequency (1d) (Average)', 'Partner (First)', 'Confidence: 7-Day (Mode)', 'Confidence: 14-Day (Mode)', 'Confidence: 30-Day (Mode)', 'Confidence: 45-Day (Mode)', 'Confidence: 1-Day (Mode)', 'Probability: 7-Day (Average)', 'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 'Probability: 45-Day (Average)', 'Probability: 1-Day (Average)', 'Last Observed', 'Indicator Type', 'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'Partners', 'incidents/events', 'Threat Actor', 'CAL Score', 'ThreatConnect Score', 'HTOC Threat Score', 'Severity', 'Explanation']

Shape: (2035, 31)


,Indicator,Observed Today (Average),Frequency (7d) (Average),Frequency (30d) (Average),Frequency (1d) (Average),Partner (First),Confidence: 7-Day (Mode),Confidence: 14-Day (Mode),Confidence: 30-Day (Mode),Confidence: 45-Day (Mode),...,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0.00,0.00,0.84,0.00,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Highly likely,...,1,0,VA,Incident:30246,NaN,810,905,177,low,Severity: low. Could not pull VT score to calc...
1,1-you.njalla.no,0.01,0.45,1.94,0.01,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,...,0,0,"NIH, VA",NaN,NaN,180,590,124,low,[2026-02-02] Severity: low. VT score not avail...
2,1.234.83.26,0.38,3.65,14.48,0.38,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,...,0,0,"CMS, FDA, HRSA, IHS, OS",NaN,NaN,550,654,199,low,[2026-02-18] Severity: low. VT score: 2. Top d...
3,1.4.195.14,0.04,0.47,2.24,0.04,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,...,1,0,"CMS, VA",NaN,NaN,180,229,134,low,Severity: low. VT score: 1. Top drivers: TOR a...
4,101.168.57.163,0.00,0.62,2.00,0.00,CMS,7-Day: Low confidence,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,...,0,0,"CMS, FDA, OS, VA",NaN,NaN,590,795,300,medium,[2026-02-18] Severity: medium. VT score: 11. T...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030,www.deepseek.com,0.10,2.16,10.59,0.04,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,...,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",NaN,NaN,180,469,336,medium,[2026-02-25] Severity: medium. VT score not av...
2031,www.deepseek.com.cdn.cloudflare.net,0.04,0.74,4.05,0.01,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,...,0,0,"CMS, DHA, IHS, NIH, VA",NaN,NaN,170,464,293,medium,Severity: medium. Could not pull VT score to c...
2032,www.prosinthecity.com,0.00,0.11,0.56,0.00,DHA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,...,0,0,"CMS, DHA, NIH, OS",Incident:6755399448004157;Incident:546481,NaN,170,363,417,medium,Severity: medium. Could not pull VT score to c...
2033,www.sthda.com,0.09,1.51,8.24,0.01,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,...,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:331131,NaN,180,368,399,medium,Severity: medium. Could not pull VT score to c...


In [6]:
# Drop specified columns
columns_to_drop = [
    'CAL Score',
    'Threat Actor',
    'incidents/events',
    'Partners',
    'False Positives',
    'Botnet Flag',
    'Observation Penalty',
    'Observation Penalty Multiplier',
    'ThreatConnect Rating',
    'Observation Yearly Count',
    'Indicator Type',
    'Last Observed'
]

print("Available columns before dropping:")
print(df_merged.columns.tolist())

# Drop columns that exist in the dataframe
cols_found = [col for col in columns_to_drop if col in df_merged.columns]
df_merged = df_merged.drop(columns=cols_found)

print(f"\nDropped columns: {cols_found}")
print(f"\nRemaining columns ({len(df_merged.columns)}): {df_merged.columns.tolist()}")
print(f"Shape: {df_merged.shape}")


Available columns before dropping:
['Indicator', 'Observed Today (Average)', 'Frequency (7d) (Average)', 'Frequency (30d) (Average)', 'Frequency (1d) (Average)', 'Partner (First)', 'Confidence: 7-Day (Mode)', 'Confidence: 14-Day (Mode)', 'Confidence: 30-Day (Mode)', 'Confidence: 45-Day (Mode)', 'Confidence: 1-Day (Mode)', 'Probability: 7-Day (Average)', 'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 'Probability: 45-Day (Average)', 'Probability: 1-Day (Average)', 'Last Observed', 'Indicator Type', 'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'Partners', 'incidents/events', 'Threat Actor', 'CAL Score', 'ThreatConnect Score', 'HTOC Threat Score', 'Severity', 'Explanation']

Dropped columns: ['CAL Score', 'Threat Actor', 'incidents/events', 'Partners', 'False Positives', 'Botnet Flag', 'Observation Penalty Multiplier', 'ThreatConnect Rating', 'Observation Yearly Count', 'Indicator Type', 'Last Ob

In [10]:
df_merged

,Indicator,Observed Today (Average),Frequency (7d) (Average),Frequency (30d) (Average),Frequency (1d) (Average),Partner (First),Confidence: 7-Day (Mode),Confidence: 14-Day (Mode),Confidence: 30-Day (Mode),Confidence: 45-Day (Mode),Confidence: 1-Day (Mode),Probability: 7-Day (Average),Probability: 14-Day (Average),Probability: 30-Day (Average),Probability: 45-Day (Average),Probability: 1-Day (Average),ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0.00,0.00,0.84,0.00,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,45-Day: Highly likely,1-Day: Low confidence,16.58%,33.45%,48.79%,52.79%,0.57%,905,177,low,Severity: low. Could not pull VT score to calc...
1,1-you.njalla.no,0.01,0.45,1.94,0.01,NIH,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,unknown,25.66%,43.13%,64.95%,50.55%,10.23%,590,124,low,[2026-02-02] Severity: low. VT score not avail...
2,1.234.83.26,0.38,3.65,14.48,0.38,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,83.04%,90.74%,95.63%,78.75%,42.34%,654,199,low,[2026-02-18] Severity: low. VT score: 2. Top d...
3,1.4.195.14,0.04,0.47,2.24,0.04,VA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,27.47%,41.21%,63.55%,57.23%,6.66%,229,134,low,Severity: low. VT score: 1. Top drivers: TOR a...
4,101.168.57.163,0.00,0.62,2.00,0.00,CMS,7-Day: Low confidence,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,1-Day: Low confidence,32.09%,69.63%,95.27%,74.49%,10.75%,795,300,medium,[2026-02-18] Severity: medium. VT score: 11. T...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030,www.deepseek.com,0.10,2.16,10.59,0.04,CMS,7-Day: Highly likely,14-Day: Highly likely,30-Day: Highly likely,45-Day: Highly likely,unknown,61.92%,71.65%,80.84%,73.63%,22.89%,469,336,medium,[2026-02-25] Severity: medium. VT score not av...
2031,www.deepseek.com.cdn.cloudflare.net,0.04,0.74,4.05,0.01,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,23.56%,32.45%,44.71%,41.54%,4.31%,464,293,medium,Severity: medium. Could not pull VT score to c...
2032,www.prosinthecity.com,0.00,0.11,0.56,0.00,DHA,7-Day: Low confidence,14-Day: Low confidence,30-Day: Low confidence,unknown,unknown,8.38%,16.41%,32.61%,33.86%,3.36%,363,417,medium,Severity: medium. Could not pull VT score to c...
2033,www.sthda.com,0.09,1.51,8.24,0.01,CMS,7-Day: Low confidence,14-Day: Low confidence,30-Day: Highly likely,45-Day: Highly likely,unknown,45.39%,58.83%,69.53%,64.02%,10.84%,368,399,medium,Severity: medium. Could not pull VT score to c...


In [11]:
# Create df_analysis from df_merged with numeric conversions for analysis

df_analysis = df_merged.copy()

# Convert probability columns to numeric (remove % and convert to float)
prob_columns = ['Probability: 1-Day (Average)', 'Probability: 7-Day (Average)', 
                'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 
                'Probability: 45-Day (Average)']

print("Converting probability columns to numeric...")
for col in prob_columns:
    if col in df_analysis.columns:
        # If it's a string with %, remove % and convert
        if df_analysis[col].dtype == 'object':
            df_analysis[col] = pd.to_numeric(df_analysis[col].str.rstrip('%'), errors='coerce')
        print(f"  ✓ {col}: {df_analysis[col].dtype}")

# Add Score_Range binning
def categorize_threat(score):
    if score >= 700:
        return 'Critical'
    elif score >= 400:
        return 'High'
    elif score >= 100:
        return 'Medium'
    else:
        return 'Low'

df_analysis['Score_Range'] = df_analysis['HTOC Threat Score'].apply(categorize_threat)

print(f"\n✅ df_analysis created: {df_analysis.shape}")
print(f"Columns: {df_analysis.columns.tolist()}")
print(f"\nThreat Score Distribution:")
print(df_analysis['Score_Range'].value_counts().sort_index())


Converting probability columns to numeric...
  ✓ Probability: 1-Day (Average): float64
  ✓ Probability: 7-Day (Average): float64
  ✓ Probability: 14-Day (Average): float64
  ✓ Probability: 30-Day (Average): float64
  ✓ Probability: 45-Day (Average): float64

✅ df_analysis created: (2035, 21)
Columns: ['Indicator', 'Observed Today (Average)', 'Frequency (7d) (Average)', 'Frequency (30d) (Average)', 'Frequency (1d) (Average)', 'Partner (First)', 'Confidence: 7-Day (Mode)', 'Confidence: 14-Day (Mode)', 'Confidence: 30-Day (Mode)', 'Confidence: 45-Day (Mode)', 'Confidence: 1-Day (Mode)', 'Probability: 7-Day (Average)', 'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 'Probability: 45-Day (Average)', 'Probability: 1-Day (Average)', 'ThreatConnect Score', 'HTOC Threat Score', 'Severity', 'Explanation', 'Score_Range']

Threat Score Distribution:
Score_Range
Critical       1
High         629
Low           40
Medium      1365
Name: count, dtype: int64


In [12]:
# SIMPLIFIED PLAIN ENGLISH ANALYSIS: Probability vs HTOC Scores & Severity

# Setup: Create df_prob if it doesn't exist
if 'df_prob' not in dir():
    df_prob = df_analysis.copy()
    # Convert probability columns to numeric if needed
    prob_columns = ['Probability: 1-Day (Average)', 'Probability: 7-Day (Average)', 
                    'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 
                    'Probability: 45-Day (Average)']
    for col in prob_columns:
        if col in df_prob.columns and df_prob[col].dtype == 'object':
            df_prob[col] = pd.to_numeric(df_prob[col].str.rstrip('%'), errors='coerce')

# Setup: Create prob_correlations if it doesn't exist
if 'prob_correlations' not in dir():
    prob_correlations = {}
    prob_columns = ['Probability: 1-Day (Average)', 'Probability: 7-Day (Average)', 
                    'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 
                    'Probability: 45-Day (Average)']
    for col in prob_columns:
        if col in df_prob.columns:
            corr = df_prob[[col, 'HTOC Threat Score']].dropna().corr().iloc[0, 1]
            prob_correlations[col.replace(' (Average)', '')] = corr

print("\n" + "=" * 100)
print("📊 SIMPLE COMPARISON: Do Higher Probabilities = Higher Threats?")
print("=" * 100)

print("""

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 QUESTION 1: Do Higher Probabilities Mean Higher HTOC Threat Scores?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ YES - There IS a clear relationship!
""")

# Compare high vs low probability
high_prob_threshold = 75
low_prob_threshold = 25

high_prob_indicators = df_prob[df_prob['Probability: 45-Day (Average)'] >= high_prob_threshold]
low_prob_indicators = df_prob[df_prob['Probability: 45-Day (Average)'] <= low_prob_threshold]
medium_prob_indicators = df_prob[(df_prob['Probability: 45-Day (Average)'] > low_prob_threshold) & 
                                  (df_prob['Probability: 45-Day (Average)'] < high_prob_threshold)]

print(f"""
📈 PROBABILITY GROUPS (Using 45-Day Forecast):

🔴 HIGH PROBABILITY (≥{high_prob_threshold}%):
   • Number of indicators: {len(high_prob_indicators):,}
   • Average threat score: {high_prob_indicators['HTOC Threat Score'].mean():.0f}
   • Typical range: {high_prob_indicators['HTOC Threat Score'].quantile(0.25):.0f} to {high_prob_indicators['HTOC Threat Score'].quantile(0.75):.0f}

🟡 MEDIUM PROBABILITY ({low_prob_threshold}-{high_prob_threshold}%):
   • Number of indicators: {len(medium_prob_indicators):,}
   • Average threat score: {medium_prob_indicators['HTOC Threat Score'].mean():.0f}
   • Typical range: {medium_prob_indicators['HTOC Threat Score'].quantile(0.25):.0f} to {medium_prob_indicators['HTOC Threat Score'].quantile(0.75):.0f}

🟢 LOW PROBABILITY (≤{low_prob_threshold}%):
   • Number of indicators: {len(low_prob_indicators):,}
   • Average threat score: {low_prob_indicators['HTOC Threat Score'].mean():.0f}
   • Typical range: {low_prob_indicators['HTOC Threat Score'].quantile(0.25):.0f} to {low_prob_indicators['HTOC Threat Score'].quantile(0.75):.0f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 WHAT THIS MEANS:
   High probability indicators score {high_prob_indicators['HTOC Threat Score'].mean() - low_prob_indicators['HTOC Threat Score'].mean():.0f} points HIGHER than low probability indicators!
   This shows your probability forecasts ARE working - they successfully identify 
   which threats are more serious.
""")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 QUESTION 2: Do Higher Probabilities Mean Higher Severity Levels?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ YES - Probability predicts severity too!
""")

# Analyze severity distribution by probability level
print("\n📊 SEVERITY BREAKDOWN BY PROBABILITY LEVEL:\n")

# Check if Severity column exists
if 'Severity' in df_prob.columns:
    print(f"HIGH PROBABILITY (≥{high_prob_threshold}%):")
    high_severity_dist = high_prob_indicators['Severity'].value_counts().sort_index()
    for sev, count in high_severity_dist.items():
        pct = 100 * count / len(high_prob_indicators)
        print(f"   • {sev}: {count:,} indicators ({pct:.1f}%)")
    
    print(f"\nMEDIUM PROBABILITY ({low_prob_threshold}-{high_prob_threshold}%):")
    med_severity_dist = medium_prob_indicators['Severity'].value_counts().sort_index()
    for sev, count in med_severity_dist.items():
        pct = 100 * count / len(medium_prob_indicators)
        print(f"   • {sev}: {count:,} indicators ({pct:.1f}%)")
    
    print(f"\nLOW PROBABILITY (≤{low_prob_threshold}%):")
    low_severity_dist = low_prob_indicators['Severity'].value_counts().sort_index()
    for sev, count in low_severity_dist.items():
        pct = 100 * count / len(low_prob_indicators)
        print(f"   • {sev}: {count:,} indicators ({pct:.1f}%)")
    
    print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 WHAT THIS MEANS:
   Higher probability indicators tend to have more severe threat classifications.
   Your probability forecasts help predict both the SCORE and SEVERITY of threats.
""")
else:
    print("   (Severity data not available in dataset)")

print("""

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 QUESTION 3: Which Probability Forecast Works Best?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Show correlation strength for each timeframe
print("📊 HOW WELL EACH FORECAST PREDICTS THREAT SCORES:\n")

prob_metrics = [
    ('7-Day Probability', prob_correlations.get('Probability: 7-Day', 0)),
    ('14-Day Probability', prob_correlations.get('Probability: 14-Day', 0)),
    ('1-Day Probability', prob_correlations.get('Probability: 1-Day', 0)),
    ('30-Day Probability', prob_correlations.get('Probability: 30-Day', 0)),
    ('45-Day Probability', prob_correlations.get('Probability: 45-Day', 0))
]

prob_metrics_sorted = sorted(prob_metrics, key=lambda x: x[1], reverse=True)

for i, (name, corr) in enumerate(prob_metrics_sorted, 1):
    stars = "⭐" * min(5, int(corr * 10))
    print(f"   {i}. {name:20s} - {stars} ({corr:.3f})")

best_metric = prob_metrics_sorted[0][0]
best_corr = prob_metrics_sorted[0][1]

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 WINNER: {best_metric} is the best predictor!
   It has the strongest relationship with actual threat scores ({best_corr:.3f})
   Use this forecast for your daily threat assessments.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 THE BOTTOM LINE (In One Sentence)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Higher probability forecasts consistently match with higher HTOC threat scores 
and more severe threat classifications - your forecasting system works!


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ SIMPLE ACTION ITEMS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Trust your probability forecasts - they DO predict threat levels

2. Prioritize indicators with probability ≥75% - they score {high_prob_indicators['HTOC Threat Score'].mean():.0f} points higher

3. Use the {best_metric.replace(' Probability', '')} forecast for daily operations

4. Lower probability (≤25%) indicators can be monitored with less urgency


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("=" * 100 + "\n")


📊 SIMPLE COMPARISON: Do Higher Probabilities = Higher Threats?


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎯 QUESTION 1: Do Higher Probabilities Mean Higher HTOC Threat Scores?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ YES - There IS a clear relationship!


📈 PROBABILITY GROUPS (Using 45-Day Forecast):

🔴 HIGH PROBABILITY (≥75%):
   • Number of indicators: 805
   • Average threat score: 403
   • Typical range: 352 to 503

🟡 MEDIUM PROBABILITY (25-75%):
   • Number of indicators: 1,174
   • Average threat score: 277
   • Typical range: 162 to 374

🟢 LOW PROBABILITY (≤25%):
   • Number of indicators: 36
   • Average threat score: 252
   • Typical range: 199 to 332

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 WHAT THIS MEANS:
   High probability indicators score 151 points HIGHER than low probability indicators!
   This shows your probabil

In [13]:
# COMPREHENSIVE BREAKDOWN: All Probability Timeframes

# Setup: Create df_prob if it doesn't exist
if 'df_prob' not in dir():
    df_prob = df_analysis.copy()
    # Convert probability columns to numeric if needed
    prob_columns = ['Probability: 1-Day (Average)', 'Probability: 7-Day (Average)', 
                    'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)', 
                    'Probability: 45-Day (Average)']
    for col in prob_columns:
        if col in df_prob.columns and df_prob[col].dtype == 'object':
            df_prob[col] = pd.to_numeric(df_prob[col].str.rstrip('%'), errors='coerce')

print("\n" + "=" * 100)
print("📊 PROBABILITY GROUPS ACROSS ALL TIMEFRAMES")
print("=" * 100)

# Define thresholds
high_threshold = 75
low_threshold = 25

# All probability columns
all_prob_cols = [
    'Probability: 1-Day (Average)',
    'Probability: 7-Day (Average)',
    'Probability: 14-Day (Average)',
    'Probability: 30-Day (Average)',
    'Probability: 45-Day (Average)'
]

timeframe_names = ['1-Day', '7-Day', '14-Day', '30-Day', '45-Day']

for col, name in zip(all_prob_cols, timeframe_names):
    if col in df_prob.columns:
        print(f"\n\n{'━' * 100}")
        print(f"📈 PROBABILITY GROUPS (Using {name} Forecast)")
        print(f"{'━' * 100}\n")
        
        # Get high, medium, low groups
        high_prob = df_prob[df_prob[col] >= high_threshold]
        medium_prob = df_prob[(df_prob[col] > low_threshold) & (df_prob[col] < high_threshold)]
        low_prob = df_prob[df_prob[col] <= low_threshold]
        
        # High probability stats
        print(f"🔴 HIGH PROBABILITY (≥{high_threshold}%):")
        print(f"   • Number of indicators: {len(high_prob):,}")
        if len(high_prob) > 0:
            print(f"   • Average threat score: {high_prob['HTOC Threat Score'].mean():.0f}")
            print(f"   • Typical range: {high_prob['HTOC Threat Score'].quantile(0.25):.0f} to {high_prob['HTOC Threat Score'].quantile(0.75):.0f}")
        else:
            print(f"   • No indicators in this category")
        
        # Medium probability stats
        print(f"\n🟡 MEDIUM PROBABILITY ({low_threshold}-{high_threshold}%):")
        print(f"   • Number of indicators: {len(medium_prob):,}")
        if len(medium_prob) > 0:
            print(f"   • Average threat score: {medium_prob['HTOC Threat Score'].mean():.0f}")
            print(f"   • Typical range: {medium_prob['HTOC Threat Score'].quantile(0.25):.0f} to {medium_prob['HTOC Threat Score'].quantile(0.75):.0f}")
        else:
            print(f"   • No indicators in this category")
        
        # Low probability stats
        print(f"\n🟢 LOW PROBABILITY (≤{low_threshold}%):")
        print(f"   • Number of indicators: {len(low_prob):,}")
        if len(low_prob) > 0:
            print(f"   • Average threat score: {low_prob['HTOC Threat Score'].mean():.0f}")
            print(f"   • Typical range: {low_prob['HTOC Threat Score'].quantile(0.25):.0f} to {low_prob['HTOC Threat Score'].quantile(0.75):.0f}")
        else:
            print(f"   • No indicators in this category")
        
        # Show the difference
        if len(high_prob) > 0 and len(low_prob) > 0:
            diff = high_prob['HTOC Threat Score'].mean() - low_prob['HTOC Threat Score'].mean()
            print(f"\n💡 DIFFERENCE: High vs Low = {diff:.0f} points")

print("\n\n" + "=" * 100)
print("✅ ANALYSIS COMPLETE: All probability timeframes analyzed")
print("=" * 100 + "\n")


📊 PROBABILITY GROUPS ACROSS ALL TIMEFRAMES


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 PROBABILITY GROUPS (Using 1-Day Forecast)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔴 HIGH PROBABILITY (≥75%):
   • Number of indicators: 49
   • Average threat score: 478
   • Typical range: 499 to 513

🟡 MEDIUM PROBABILITY (25-75%):
   • Number of indicators: 794
   • Average threat score: 396
   • Typical range: 347 to 499

🟢 LOW PROBABILITY (≤25%):
   • Number of indicators: 1,140
   • Average threat score: 273
   • Typical range: 156 to 373

💡 DIFFERENCE: High vs Low = 205 points


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 PROBABILITY GROUPS (Using 7-Day Forecast)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔴 HIGH PROBABILITY (≥75%):
   • Number of indicators

In [15]:
# FREQUENCY AVERAGES BY PROBABILITY GROUPS (Same indicator counts)

# Setup: Ensure df_prob exists
if 'df_prob' not in dir():
    df_prob = df_analysis.copy()
    prob_columns = ['Probability: 1-Day (Average)', 'Probability: 7-Day (Average)',
                    'Probability: 14-Day (Average)', 'Probability: 30-Day (Average)',
                    'Probability: 45-Day (Average)']
    for col in prob_columns:
        if col in df_prob.columns and df_prob[col].dtype == 'object':
            df_prob[col] = pd.to_numeric(df_prob[col].str.rstrip('%'), errors='coerce')

# Frequency columns to summarize
freq_cols = [
    'Frequency (1d) (Average)',
    'Frequency (7d) (Average)',
    'Frequency (30d) (Average)'
]

existing_freq_cols = [c for c in freq_cols if c in df_prob.columns]

print("\n" + "=" * 100)
print("📊 FREQUENCY AVERAGES BY PROBABILITY GROUPS")
print("=" * 100)

if not existing_freq_cols:
    print("No frequency columns found. Please confirm the column names in df_prob.")
else:
    # Thresholds match probability grouping
    high_threshold = 75
    low_threshold = 25

    all_prob_cols = [
        'Probability: 1-Day (Average)',
        'Probability: 7-Day (Average)',
        'Probability: 14-Day (Average)',
        'Probability: 30-Day (Average)',
        'Probability: 45-Day (Average)'
    ]

    timeframe_names = ['1-Day', '7-Day', '14-Day', '30-Day', '45-Day']

    def print_freq_stats(label, group_df):
        print(f"{label} ({len(group_df):,} indicators)")
        if len(group_df) == 0:
            print("   • No indicators in this category")
            return
        for c in existing_freq_cols:
            avg_val = group_df[c].mean()
            min_val = group_df[c].min()
            max_val = group_df[c].max()
            print(f"   • {c}: avg {avg_val:.2f}, min {min_val:.2f}, max {max_val:.2f}")

    for col, name in zip(all_prob_cols, timeframe_names):
        if col in df_prob.columns:
            print(f"\n\n{'━' * 100}")
            print(f"📈 FREQUENCY AVERAGES (Using {name} Probability Groups)")
            print(f"{'━' * 100}\n")

            high_prob = df_prob[df_prob[col] >= high_threshold]
            medium_prob = df_prob[(df_prob[col] > low_threshold) & (df_prob[col] < high_threshold)]
            low_prob = df_prob[df_prob[col] <= low_threshold]

            print_freq_stats(f"🔴 HIGH PROBABILITY (≥{high_threshold}%)", high_prob)
            print("")
            print_freq_stats(f"🟡 MEDIUM PROBABILITY ({low_threshold}-{high_threshold}%)", medium_prob)
            print("")
            print_freq_stats(f"🟢 LOW PROBABILITY (≤{low_threshold}%)", low_prob)

print("\n\n" + "=" * 100)
print("✅ FREQUENCY SUMMARY COMPLETE")
print("=" * 100 + "\n")



📊 FREQUENCY AVERAGES BY PROBABILITY GROUPS


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 FREQUENCY AVERAGES (Using 1-Day Probability Groups)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔴 HIGH PROBABILITY (≥75%) (49 indicators)
   • Frequency (1d) (Average): avg 0.95, min 0.87, max 0.99
   • Frequency (7d) (Average): avg 6.47, min 4.91, max 6.88
   • Frequency (30d) (Average): avg 21.93, min 20.21, max 25.84

🟡 MEDIUM PROBABILITY (25-75%) (794 indicators)
   • Frequency (1d) (Average): avg 0.64, min 0.00, max 1.00
   • Frequency (7d) (Average): avg 4.62, min 0.08, max 6.81
   • Frequency (30d) (Average): avg 17.92, min 0.12, max 25.52

🟢 LOW PROBABILITY (≤25%) (1,140 indicators)
   • Frequency (1d) (Average): avg 0.03, min 0.00, max 0.28
   • Frequency (7d) (Average): avg 0.65, min 0.00, max 3.66
   • Frequency (30d) (Average): avg 3.01, min 0.00, max 17.88


━━━━━━━━━━━